<a href="https://colab.research.google.com/github/priyansh-commits/practice/blob/main/grad_boost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
df = pd.DataFrame([[165,137,472,192],[101,92,250,144],[29,127,201,91]],columns=['R&D','Ops','Marketing','Profit'])
df

,R&D,Ops,Marketing,Profit
0,165,137,472,192
1,101,92,250,144
2,29,127,201,91


In [3]:
df['f0']=df['Profit'].mean()
df

,R&D,Ops,Marketing,Profit,f0
0,165,137,472,192,142.333333
1,101,92,250,144,142.333333
2,29,127,201,91,142.333333


In [ ]:
# 1. Initialize: F_0(x) = argmin_c sum(L(y_i, c))

# 2. For t = 1 to T:
#    a. Compute pseudo-residuals:
#       r_i = -dL(y_i, F_{t-1}(x_i)) / dF_{t-1}(x_i)
#    b. Fit a tree h_t to the residuals r_i
#    c. Find optimal step size:
#       gamma_t = argmin_gamma sum(L(y_i, F_{t-1}(x_i) + gamma * h_t(x_i)))
#    d. Update:
#       F_t(x) = F_{t-1}(x) + learning_rate * gamma_t * h_t(x)

# 3. Final prediction: F_T(x)

In [5]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

# ==========================================
# 1. Create a small dataset
# ==========================================

X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
    [6]
])

y = np.array([2, 4, 5, 7, 8, 10])

# Learning rate
learning_rate = 0.5

# Number of trees
n_estimators = 30


# ==========================================
# 2. Initial Prediction F0
# ==========================================

# For squared error, F0 is mean of y
initial_prediction = np.mean(y)

# Every point initially gets the same prediction
predictions = np.full(len(y), initial_prediction)

print("=" * 70)
print("INITIAL MODEL F0")
print("=" * 70)

print(f"\nInitial Prediction (Mean of y): {initial_prediction:.2f}")

# Initial residuals
residuals = y - predictions

df = pd.DataFrame({
    "X": X.flatten(),
    "Actual_y": y,
    "Prediction_F0": predictions,
    "Residual": residuals
})

print(df.to_string(index=False))


# ==========================================
# 3. Gradient Boosting Iterations
# ==========================================

trees = []

for m in range(n_estimators):

    print("\n" + "=" * 70)
    print(f"DECISION TREE {m+1}")
    print("=" * 70)

    # --------------------------------------
    # Show predictions BEFORE this tree
    # --------------------------------------

    print("\nPredictions BEFORE Tree:")
    print(predictions)

    # Calculate residuals
    # For MSE:
    # Negative Gradient = y - prediction

    residuals = y - predictions

    print("\nResiduals BEFORE Tree:")
    print(residuals)


    # --------------------------------------
    # Train DT on residuals
    # --------------------------------------

    tree = DecisionTreeRegressor(
        max_depth=2,
        random_state=42
    )

    tree.fit(X, residuals)

    # Tree predicts correction
    tree_prediction = tree.predict(X)

    print("\nTree Prediction (Correction):")
    print(tree_prediction)


    # --------------------------------------
    # Update predictions
    # --------------------------------------

    predictions = predictions + learning_rate * tree_prediction

    print("\nPredictions AFTER Tree:")
    print(predictions)


    # --------------------------------------
    # Calculate new residuals
    # --------------------------------------

    new_residuals = y - predictions

    print("\nResiduals AFTER Tree:")
    print(new_residuals)


    # Display everything in table
    df = pd.DataFrame({
        "X": X.flatten(),
        "Actual_y": y,
        "Old_Prediction": predictions - learning_rate * tree_prediction,
        "Old_Residual": residuals,
        "Tree_Correction": tree_prediction,
        "New_Prediction": predictions,
        "New_Residual": new_residuals
    })

    print("\nComplete Step Table:")
    print(df.to_string(index=False))

    # Save tree
    trees.append(tree)


# ==========================================
# 4. Final Results
# ==========================================

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

final_df = pd.DataFrame({
    "X": X.flatten(),
    "Actual_y": y,
    "Final_Prediction": predictions,
    "Final_Residual": y - predictions
})

print(final_df.to_string(index=False))

INITIAL MODEL F0

Initial Prediction (Mean of y): 6.00
 X  Actual_y  Prediction_F0  Residual
 1         2            6.0      -4.0
 2         4            6.0      -2.0
 3         5            6.0      -1.0
 4         7            6.0       1.0
 5         8            6.0       2.0
 6        10            6.0       4.0

DECISION TREE 1

Predictions BEFORE Tree:
[6. 6. 6. 6. 6. 6.]

Residuals BEFORE Tree:
[-4. -2. -1.  1.  2.  4.]

Tree Prediction (Correction):
[-4.  -1.5 -1.5  1.5  1.5  4. ]

Predictions AFTER Tree:
[4.   5.25 5.25 6.75 6.75 8.  ]

Residuals AFTER Tree:
[-2.   -1.25 -0.25  0.25  1.25  2.  ]

Complete Step Table:
 X  Actual_y  Old_Prediction  Old_Residual  Tree_Correction  New_Prediction  New_Residual
 1         2             6.0          -4.0             -4.0            4.00         -2.00
 2         4             6.0          -2.0             -1.5            5.25         -1.25
 3         5             6.0          -1.0             -1.5            5.25         -0.25
 4 